In [0]:
import urllib.request
import os
from datetime import datetime, timedelta, timezone

# Configuration
VOLUME_PATH = "/Volumes/gharchive_dev/raw/files"
BASE_URL = "https://data.gharchive.org"

# Current UTC time (use previous hour since current hour data isn't available yet)
now_utc = datetime.now(timezone.utc)
prev_hour = now_utc - timedelta(hours=1)
default_date = prev_hour.strftime("%Y-%m-%d")
default_hour = str(prev_hour.hour)

print(f"Current UTC time: {now_utc.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Default download target: {default_date} hour {default_hour}")

# Download files (configurable via widgets)
dbutils.widgets.text("date", default_date, "Date (YYYY-MM-DD)")
dbutils.widgets.text("hours", default_hour, "Hours (comma-separated, e.g. 0,1,2 or 11)")

target_date = dbutils.widgets.get("date")
hours = [int(h.strip()) for h in dbutils.widgets.get("hours").split(",")]

print(f"\nDownloading GH Archive for {target_date}, hours: {hours}")
print(f"Target volume: {VOLUME_PATH}")

downloaded_files = []
for hour in hours:
    filename = f"{target_date}-{hour}.json.gz"
    url = f"{BASE_URL}/{filename}"
    volume_dest = f"{VOLUME_PATH}/{filename}"

    # Check if already downloaded
    try:
        dbutils.fs.ls(volume_dest)
        print(f"SKIP: {filename} already exists in volume")
        downloaded_files.append(filename)
        continue
    except Exception:
        pass

    print(f"Downloading: {url}")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            data = response.read()

        # Write directly to Volume FUSE path (no local /tmp needed)
        with open(volume_dest, "wb") as out:
            out.write(data)
        print(f"SUCCESS: {filename} -> {volume_dest} ({len(data):,} bytes)")
        downloaded_files.append(filename)
    except Exception as e:
        print(f"FAILED: {filename} - {e}")

# Fail explicitly if no files were downloaded
if not downloaded_files:
    raise RuntimeError(f"No files downloaded for {target_date}, hours {hours}. GH Archive data may not be available yet.")

# Pass filename pattern to downstream tasks (bronze notebook)
if len(downloaded_files) == 1:
    filename_pattern = downloaded_files[0]
else:
    filename_pattern = f"{target_date}-*.json.gz"

dbutils.jobs.taskValues.set(key="filename", value=filename_pattern)
print(f"\nTask value set: filename = {filename_pattern}")

# List files in volume
print("\nFiles in volume:")
for f in dbutils.fs.ls(VOLUME_PATH):
    print(f"  {f.name} ({f.size:,} bytes)")